In [4]:
import pandas as pd
import numpy as np

# Load the dataset
print("📊 Loading dataset...")
df = pd.read_csv('dataset/feature_data_v2.csv')

print(f"Dataset shape before: {df.shape}")

# 🔹 1. Route Identity Features
print("\n🔹 1. Creating Route Identity Features...")

# Create stable route representation (origin + "_" + destination)
df['route_id'] = df['origin_city'] + "_" + df['destination_city']

# Identify return routes (origin == destination)
# Note: In this dataset, origin and destination are different cities, but we create for completeness
df['is_return_route'] = (df['origin_city'] == df['destination_city']).astype(int)
print(f"   - Return routes found: {df['is_return_route'].sum()}")

# 🔹 2. Route Frequency & Popularity
print("\n🔹 2. Creating Route Frequency & Popularity Features...")

# Calculate historical frequency for each route
route_counts = df['route_id'].value_counts()
df['route_frequency'] = df['route_id'].map(route_counts)

# Categorize into popularity buckets
def categorize_popularity(freq):
    if freq <= 10:
        return 'rare'
    elif freq <= 100:
        return 'common'
    else:
        return 'popular'

df['route_popularity_bucket'] = df['route_frequency'].apply(categorize_popularity)
print(f"   - Route distribution: {df['route_popularity_bucket'].value_counts().to_dict()}")

# 🔹 3. Historical Route Performance (SAFE AGGREGATION - NO delivery_time)
print("\n🔹 3. Creating Historical Route Performance Features (NO delivery_time leakage)...")

# Group by route and calculate distance statistics
route_stats = df.groupby('route_id')['distance_km'].agg([
    ('route_avg_distance', 'mean'),
    ('route_std_distance', 'std')
]).fillna(0)  # Fill NaN std with 0 for routes with single observation

# Merge back to main dataframe
df = df.merge(route_stats, on='route_id', how='left')

# 🔹 4. Distance Normalization per Route
print("\n🔹 4. Creating Distance Normalization Features...")

# Calculate distance relative to route average (captures detours/anomalies)
df['distance_vs_route_avg'] = df['distance_km'] / df['route_avg_distance']

# Handle edge cases where route_avg_distance is 0 (shouldn't happen with real data)
df['distance_vs_route_avg'] = df['distance_vs_route_avg'].replace([np.inf, -np.inf], 1.0)

print(f"   - Distance ratio range: [{df['distance_vs_route_avg'].min():.2f}, {df['distance_vs_route_avg'].max():.2f}]")

# 🔹 5. Directional Complexity Proxy
print("\n🔹 5. Creating Directional Complexity Features...")

# Number of unique vehicle types used on each route
vehicle_diversity = df.groupby('route_id')['vehicle_type'].nunique().reset_index()
vehicle_diversity.columns = ['route_id', 'city_pair_complexity']

# Merge back to main dataframe
df = df.merge(vehicle_diversity, on='route_id', how='left')
print(f"   - Average vehicle types per route: {df['city_pair_complexity'].mean():.2f}")

# 🔹 6. Long-Haul Route Indicator
print("\n🔹 6. Creating Long-Haul Route Indicator...")

# Calculate 75th percentile of route average distances
long_haul_threshold = df['route_avg_distance'].quantile(0.75)
df['is_long_route'] = (df['route_avg_distance'] > long_haul_threshold).astype(int)

print(f"   - Long-haul threshold: {long_haul_threshold:.1f} km")
print(f"   - Long-haul routes: {df['is_long_route'].sum()} ({df['is_long_route'].mean()*100:.1f}%)")

# 🧪 VALIDATION
print("\n" + "="*50)
print("🧪 VALIDATION RESULTS")
print("="*50)

print(f"Dataset shape after: {df.shape}")
print(f"New features added: {set(df.columns) - set(pd.read_csv('dataset/feature_data_v2.csv').columns)}")

# Check for missing values
missing_pct = df.isnull().sum() / len(df) * 100
missing_features = missing_pct[missing_pct > 0]
if len(missing_features) == 0:
    print("✅ No missing values in new features")
else:
    print(f"⚠️  Features with missing values:\n{missing_features}")

# Top 5 most frequent routes
print(f"\nTop 5 most frequent routes:")
top_routes = df[['route_id', 'route_frequency']].drop_duplicates().sort_values('route_frequency', ascending=False).head()
for idx, row in top_routes.iterrows():
    print(f"   {row['route_id']}: {int(row['route_frequency'])} trips")

# Verify no data leakage
if 'delivery_time_hours' in df.columns:
    # Check that we didn't accidentally use delivery_time in feature creation
    new_cols = ['route_avg_distance', 'route_std_distance', 'distance_vs_route_avg',
                'city_pair_complexity', 'is_long_route']
    for col in new_cols:
        # These shouldn't correlate perfectly with delivery_time if we avoided leakage
        corr = df[col].corr(df['delivery_time_hours'])
        print(f"   - Correlation ({col} with delivery_time): {corr:.3f}")

# 💾 Save output
output_file = 'dataset/feature_data_v3.csv'
df.to_csv(output_file, index=False)
print(f"\n💾 Dataset saved as: {output_file}")

# Final confirmation
print("\n" + "="*50)
print("✅ BLOCK 4B COMPLETE: Spatial features engineered successfully")
print("="*50)
print("\n📈 FEATURE SUMMARY:")
print(f"   - Original columns: {len(pd.read_csv('dataset/feature_data_v2.csv').columns)}")
print(f"   - New columns: {len(df.columns) - len(pd.read_csv('dataset/feature_data_v2.csv').columns)}")
print(f"   - Total columns: {len(df.columns)}")
print(f"   - Rows: {len(df)}")
print(f"   - Unique routes: {df['route_id'].nunique()}")

📊 Loading dataset...
Dataset shape before: (69926, 26)

🔹 1. Creating Route Identity Features...
   - Return routes found: 0

🔹 2. Creating Route Frequency & Popularity Features...
   - Route distribution: {'popular': 69926}

🔹 3. Creating Historical Route Performance Features (NO delivery_time leakage)...

🔹 4. Creating Distance Normalization Features...
   - Distance ratio range: [1.00, 1.00]

🔹 5. Creating Directional Complexity Features...
   - Average vehicle types per route: 5.00

🔹 6. Creating Long-Haul Route Indicator...
   - Long-haul threshold: 808.4 km
   - Long-haul routes: 17101 (24.5%)

🧪 VALIDATION RESULTS
Dataset shape after: (69926, 35)
New features added: {'is_long_route', 'route_avg_distance', 'city_pair_complexity', 'distance_vs_route_avg', 'route_std_distance', 'route_frequency', 'route_popularity_bucket', 'is_return_route', 'route_id'}
✅ No missing values in new features

Top 5 most frequent routes:
   tashkent_samarkand: 1555 trips
   tashkent_bukhara: 1555 trips

/home/bobur/AI_Project/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3065: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/bobur/AI_Project/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3066: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]



💾 Dataset saved as: dataset/feature_data_v3.csv

✅ BLOCK 4B COMPLETE: Spatial features engineered successfully

📈 FEATURE SUMMARY:
   - Original columns: 26
   - New columns: 9
   - Total columns: 35
   - Rows: 69926
   - Unique routes: 45
